In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DB_PATH = Path("data/plant.duckdb")
conn = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected to {DB_PATH}")

Connected to data/plant.duckdb


In [2]:
# all tables and row counts
tables = conn.execute("SHOW TABLES").fetchdf()
print(tables)

for table in tables["name"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count} rows")

               name
0  plant_companions
1    plant_diseases
2       plant_pests
3       plant_types
4   plant_varieties
  plant_companions: 409 rows
  plant_diseases: 219 rows
  plant_pests: 219 rows
  plant_types: 73 rows
  plant_varieties: 73 rows


In [3]:
# plant types
conn.execute("SELECT * FROM plant_types ORDER BY category, name").fetchdf()

,plant_type_id,name,category
0,6bf1eb4a-7096-42e6-be0b-ee4078758f76,Basil,herbs
1,4c7f4a75-3141-4796-96ba-66f6213bc704,Bay Laurel,herbs
2,e2de69e9-0764-446e-b9ee-4c367bcadcaf,Borage,herbs
3,c89c3d95-8624-4a12-9fa7-00fd243ca760,Chervil,herbs
4,159f1922-0fff-418c-bdbb-3161afd78dbc,Chives,herbs
...,...,...,...
68,313bf9b9-92dd-45b9-97e9-3363b1955906,Tomatillos,vegetables
69,1e5fde50-9fe5-4ac2-b800-88834cdbd130,Tomatoes,vegetables
70,9e3e6229-aafa-4221-be1b-c167171aa1d1,Turnips,vegetables
71,509647c1-32c2-4f2a-bf46-431428da3b6a,Winter Squash,vegetables


In [4]:
# varieties — growing data
conn.execute("""
    SELECT
        pt.name,
        pt.category,
        pv.sun_tolerance,
        pv.water_required,
        pv.soil_n, pv.soil_p, pv.soil_k,
        pv.days_to_harvest,
        pv.spacing_inches,
        pv.temp_min_air_f,
        pv.temp_min_ground_f,
        pv.height_inches_estimate,
        pv.outdoor_sow_date_range,
        pv.indoor_sow_weeks_before_frost
    FROM plant_types pt
    JOIN plant_varieties pv ON pt.plant_type_id = pv.plant_type_id
    ORDER BY pt.category, pt.name
""").fetchdf()

,name,category,sun_tolerance,water_required,soil_n,soil_p,soil_k,days_to_harvest,spacing_inches,temp_min_air_f,temp_min_ground_f,height_inches_estimate,outdoor_sow_date_range,indoor_sow_weeks_before_frost
0,Basil,herbs,full_sun,medium,0.40,0.30,0.30,63,15.0,60.0,60.0,18.0,After last frost,6
1,Bay Laurel,herbs,partial_shade,low,0.30,0.20,0.30,730,42.0,40.0,40.0,120.0,n/a,8
2,Borage,herbs,partial_shade,medium,0.70,0.50,0.60,60,15.0,50.0,60.0,36.0,After the danger of frost has passed,6
3,Chervil,herbs,partial_shade,medium,0.30,0.30,0.20,50,9.0,45.0,45.0,18.0,Late spring,4
4,Chives,herbs,partial_shade,medium,0.80,0.50,0.50,60,10.0,45.0,40.0,18.0,March 15 - May 1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,Tomatillos,vegetables,full_sun,medium,0.60,0.60,0.65,85,30.0,60.0,55.0,42.0,Transplant when soil is 60°F,7
69,Tomatoes,vegetables,full_sun,high,0.85,0.70,0.80,80,21.0,60.0,60.0,48.0,Transplant when soil is 60°F,7
70,Turnips,vegetables,partial_shade,medium,0.60,0.50,0.50,30,5.0,45.0,40.0,15.0,March 15 - April 1,0
71,Winter Squash,vegetables,full_sun,high,0.85,0.75,0.80,95,36.0,70.0,70.0,18.0,When soil is at least 70°F,4


In [5]:
# companions and antagonists for a specific plant
PLANT = "tomatoes"  # change me

conn.execute("""
    SELECT pc.companion_name, pc.relationship
    FROM plant_companions pc
    JOIN plant_types pt ON pc.plant_type_id = pt.plant_type_id
    WHERE LOWER(pt.name) = LOWER(?)
    ORDER BY pc.relationship, pc.companion_name
""", [PLANT]).fetchdf()

,companion_name,relationship
0,Corn,antagonist
1,kohlrabi,antagonist
2,potatoes,antagonist
3,Basil,companion
4,carrots,companion
5,onions,companion


In [6]:
# pests for a specific plant
conn.execute("""
    SELECT pp.pest_name, pp.symptoms, pp.treatment
    FROM plant_pests pp
    JOIN plant_types pt ON pp.plant_type_id = pt.plant_type_id
    WHERE LOWER(pt.name) = LOWER(?)
""", [PLANT]).fetchdf()

,pest_name,symptoms,treatment
0,Tomato Hornworms,Chewed leaves,Hand-pick or use Bt spray
1,Aphids,Yellow leaves,Insecticidal soap or neem oil
2,Whiteflies,Sooty mold growth,Use yellow sticky traps


In [7]:
# diseases for a specific plant
conn.execute("""
    SELECT pd.disease_name, pd.symptoms, pd.treatment
    FROM plant_diseases pd
    JOIN plant_types pt ON pd.plant_type_id = pt.plant_type_id
    WHERE LOWER(pt.name) = LOWER(?)
""", [PLANT]).fetchdf()

,disease_name,symptoms,treatment
0,Late Blight,Water-soaked lesions,"Remove affected plants, apply fungicides"
1,Blossom End Rot,Black sunken spots,Ensure even watering and calcium
2,Early Blight,Brown spots,Use copper fungicides and rotate crops


In [8]:
# full plant detail — everything for one plant
conn.execute("""
    SELECT
        pt.name, pt.category,
        pv.*
    FROM plant_types pt
    JOIN plant_varieties pv ON pt.plant_type_id = pv.plant_type_id
    WHERE LOWER(pt.name) = LOWER(?)
""", [PLANT]).fetchdf().T  # transpose for readability

,0
name,Tomatoes
category,vegetables
variety_id,106bfb59-e865-476e-aa4f-cbda4dcff63e
plant_type_id,1e5fde50-9fe5-4ac2-b800-88834cdbd130
variety_name,Common
plant_category,vegetables
sun_tolerance,full_sun
water_required,high
soil_n,0.85
soil_p,0.7


In [9]:
conn.close()

In [10]:
conn

In [11]:
conn.close()